# Text Simplification Model Training & Evaluation Pipeline

A comprehensive, production-grade, and heavily annotated training pipeline for fine-tuning
sequence-to-sequence transformer models (such as BART or T5) on English text simplification.

Converted from `text_simplification_pipeline.py` into a notebook, organized by the original
script's `PART` sections so each stage can be run and inspected independently.

---

### Thesis context & theoretical design choices
*(based on expose-draft2.md / roadmap.md)*

**1. WikiSmall Omission (Dataset Critique)**
During the exploratory phase, the standard WikiSmall corpus (`cestwc/adapted-wikismall`) was
analyzed and found to contain pervasive named-entity anonymization tags (e.g., `PERSON@1`,
`LOCATION@1`, `NUMBER@1`) baked directly into its original 2010 corpus construction. While
useful for historical alignment research, these placeholder tags make the corpus unusable for
the objective of simplifying real-world web text via a browser extension — translating "John
Doe visited Paris" into "PERSON@1 visited LOCATION@1" ruins the user experience. Consequently,
WikiSmall is rejected in favor of training exclusively on the clean splits of WikiLarge
(`eilamc14/wikilarge-clean`).

**2. Deterministic Preprocessing and Backend Alignment**
To prevent training/inference divergence, the preprocessing in this pipeline uses the exact
same whitespace-collapsing regular expression logic as the FastAPI backend's
`extract_and_clean()` utility. Text is normalized prior to tokenization, ensuring the model is
exposed to identical distributions during both fine-tuning and browser-based real-time
inference.

**3. Mojibake Filter (Data Cleaning)**
WikiLarge's original alignment pipeline left a trail of garbled multi-byte UTF-8 sequences
(mojibake), where accented characters and en-dashes were split by stray spaces (e.g.,
`1809 â '' 11` instead of `1809 - 11`). Traditional automatic repair libraries (like `ftfy`)
fail to resolve this because the byte sequence is non-contiguous. This notebook implements a
deterministic regex-based mojibake scanner that detects these garbled patterns and drops
affected rows (approx. 5% of the training set), while carefully preserving valid, clean
accented text (e.g., "São Tomé", "Atlético Madrid").

**4. Scope-Tiered Strategy**
To avoid wasting valuable GPU resources, an explicit `SCOPE` flag selects one of three tiers:
`"prove_loop"` runs an end-to-end training loop on a tiny slice of the dataset (200 training
rows, 1 epoch) to prove that tokenization, collation, backpropagation, and logging function
flawlessly; `"reduced"` runs a real training loop on a bounded ~20k-row slice (2 epochs) that
finishes in well under an hour, for a real-ish check of model quality; `"full"` trains on the
entire dataset before initiating full-scale, multi-hour training runs.

**5. Multi-Strategy Decoding and Inference Sandbox**
Changing the decoding strategy during generation shifts the trade-off between semantic
preservation (staying close to the source) and simplicity gain (making active changes). Three
distinct decoding pathways are implemented and documented — Greedy, Beam Search, and
Nucleus/Top-P Sampling — including an on-the-fly "Playground" block to evaluate these
strategies side-by-side.

**6. Significance and Evaluation**
Includes stubs and calculations for BLEU, SARI, and FKGL, as well as a mathematical workflow
for running a paired bootstrap resampling test on the test split to verify that model
improvements are statistically significant.

In [ ]:
# Colab only: mount Drive so checkpoints/results (RESULTS_OUTPUT) survive the session.
from google.colab import drive
drive.mount('/content/drive')

## Setup: Imports, Logging & Hardware Check

In [ ]:
import os
import re
import sys
import logging
import argparse
import random
import numpy as np
import torch
from typing import Dict, List, Tuple

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Bold message text only (not the timestamp/level prefix), so step banners, demo outputs
# and evaluation results stand out from routine HTTP/debug noise.
BOLD = "\033[1m"
RESET = "\033[0m"

def log_highlight(msg: str):
    msg = f"{BOLD}{msg}{RESET}"
    logger.info(msg)
    print(msg)

In [ ]:
# Not preinstalled on Colab (locally pinned in requirements.txt), so install on demand.
import importlib
import subprocess

def _ensure_installed(import_name: str, pip_spec: str):
    try:
        importlib.import_module(import_name)
    except ImportError:
        logger.info(f"'{import_name}' not importable -- installing via pip ({pip_spec})...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], check=True)

_ensure_installed("sacrebleu", "sacrebleu")
_ensure_installed("easse", "git+https://github.com/feralvam/easse.git@master#egg=easse")
_ensure_installed("lens", "lens-metric")
_ensure_installed("bert_score", "bert-score")


In [ ]:
# HF_TOKEN: locally from the environment or a cached `huggingface-cli login`; on Colab from
# Secrets. `userdata` is only touched on Colab and only if the token isn't already set.
if "HF_TOKEN" not in os.environ:
    try:
        from google.colab import userdata
    except ImportError:
        logger.info(
            "HF_TOKEN not set and not running on Colab -- relying on a locally cached "
            "token from `huggingface-cli login`, if any."
        )
    else:
        try:
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
            logger.info("HF_TOKEN set from Colab Secrets.")
        except Exception as e:
            logger.warning(
                f"Running on Colab but couldn't read the HF_TOKEN secret ({e}). Add it "
                "under the key icon in the left sidebar and enable notebook access for it."
            )

In [ ]:
# set_seed() covers random, NumPy and torch (CPU + CUDA), but not torch's separate MPS generator.
from transformers import set_seed

SEED = 42
set_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)
logger.info(f"Global random seed set to {SEED} for reproducibility.")

In [ ]:
# Trainer/accelerate use MPS on Apple Silicon regardless of this variable, so it must include
# MPS or model.to(device) outside the Trainer (e.g. the inference sandbox) silently runs on CPU.
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
logger.info(f"System Hardware Check: Using device [{device.upper()}]")
if device == "cuda":
    logger.info(f"GPU Model: {torch.cuda.get_device_name(0)}")

In [ ]:
# Fresh Colab runtimes have no NLTK data; easse's FKGL/SARI tokenization needs 'punkt_tab'.
import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    logger.info("Downloading 'punkt_tab' NLTK resource...")
    nltk.download('punkt_tab', quiet=True)

## Functions and Code


### Deterministic Preprocessing & Data Cleaning

Defines the whitespace-collapsing normalization (matching the FastAPI backend's
`extract_and_clean()`) and the mojibake detector/filter used to clean the WikiLarge splits.

In [ ]:
def collapse_whitespace(text: str) -> str:
    """
    Standardizes whitespace formatting across all text nodes.
    This logic matches the backend's extract_and_clean() exactly.
    """
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def contains_mojibake(text: str) -> bool:
    """
    Scans a text string for garbled multi-byte UTF-8 sequences (mojibake)
    split by stray spaces, which are common artifacts in WikiLarge.

    Examples of garbled strings:
      - "1809 â '' 11" (corrupted en-dash)
      - "Ã ©" (split e-acute)

    We drop these rows during dataset preparation. Clean, valid accents
    like "Príncipe", "São Tomé", or "Atlético" do NOT trigger this filter.
    """
    if not text:
        return False

    # Signatures inherited from web scraping/alignment
    mojibake_patterns = [
        r"â\s*''",       # Broken en-dash or punctuation
        r"â\s*€\s*™",    # Broken single curly quote
        r"Ã\s*©",        # Broken e-acute (é)
        r"Ã\s*¹",        # Broken u-grave (ù)
        r"Ã\s*¶",        # Broken o-umlaut (ö)
        r"â\s*€\s*œ",    # Broken opening double quote
        r"â\s*€\s*",    # Broken closing double quote
    ]

    for pattern in mojibake_patterns:
        if re.search(pattern, text):
            return True

    return False

In [ ]:
def clean_and_filter_split(
    sources: List[str],
    targets: List[str],
    split_name: str
) -> Tuple[List[str], List[str]]:
    """
    Processes a dataset split by:
      1. Collapsing whitespace deterministically.
      2. Scanning and filtering out rows containing corrupted mojibake sequences.
      3. Returning clean, parallel arrays of inputs and references.
    """
    clean_sources = []
    clean_targets = []
    skipped_count = 0

    for src, tgt in zip(sources, targets):
        cleaned_src = collapse_whitespace(src)
        cleaned_tgt = collapse_whitespace(tgt)

        if contains_mojibake(cleaned_src) or contains_mojibake(cleaned_tgt):
            skipped_count += 1
            continue

        clean_sources.append(cleaned_src)
        clean_targets.append(cleaned_tgt)

    total = len(sources)
    pct_dropped = (skipped_count / total) * 100 if total > 0 else 0
    logger.info(
        f"[{split_name.upper()} SPLIT] Filtered out {skipped_count}/{total} "
        f"rows containing mojibake ({pct_dropped:.2f}% dropped). "
        f"Remaining parallel pairs: {len(clean_sources)}"
    )

    return clean_sources, clean_targets

### Dataset Simulation / Loading

Ready for Hugging Face Hub integration. Currently simulates raw dataset slices (train,
validation, test) and applies the deterministic cleaning/filtering from Part 1.

In [ ]:
def load_text_simplification_dataset(
    scope: str = "full",
    structure: str = "sentence"
) -> Dict[str, Dict[str, List[str]]]:
    """
    Loads and prepares dataset splits from a WikiLarge corpus (source/target columns,
    train/validation/test splits), then applies the deterministic whitespace/mojibake
    cleaning from Part 1.

    `structure` selects the corpus layout:
      "sentence": the clean 'eilamc14/wikilarge-clean' flat source/target sentence pairs.
      "document": D-Wikipedia -- full source/simplified Wikipedia article pairs (see
                  load_d_wikipedia_dataset below).
    """
    from datasets import load_dataset

    if scope not in ("prove_loop", "reduced", "full"):
        raise ValueError(f"Unknown scope: {scope!r}. Expected 'prove_loop', 'reduced', or 'full'.")
    if structure not in ("sentence", "document"):
        raise ValueError(f"Unknown structure: {structure!r}. Expected 'sentence' or 'document'.")

    if structure == "document":
        return load_d_wikipedia_dataset(scope=scope)

    logger.info(f"Initializing dataset ingestion (scope={scope!r}). Target: 'eilamc14/wikilarge-clean'...")

    # Scope tiers:
    #   "prove_loop": tiny slice to check the pipeline end to end.
    #   "reduced":    ~20k rows; real epochs and eval in under an hour, to check model quality
    #                 before the multi-hour full run.
    #   "full":       the entire ~124k-row training set.
    if scope == "prove_loop":
        train_split, val_split, test_split = "train[:200]", "validation[:50]", "test[:20]"
    elif scope == "reduced":
        train_split, val_split, test_split = "train[:20000]", "validation[:200]", "test[:50]"
    else:
        train_split, val_split, test_split = "train", "validation", "test"

    raw_train = load_dataset("eilamc14/wikilarge-clean", split=train_split)
    raw_val = load_dataset("eilamc14/wikilarge-clean", split=val_split)
    raw_test = load_dataset("eilamc14/wikilarge-clean", split=test_split)

    train_src, train_tgt = clean_and_filter_split(raw_train["source"], raw_train["target"], "train")
    val_src, val_tgt = clean_and_filter_split(raw_val["source"], raw_val["target"], "validation")
    test_src, test_tgt = clean_and_filter_split(raw_test["source"], raw_test["target"], "test")

    return {
        "train": {"source": train_src, "target": train_tgt},
        "validation": {"source": val_src, "target": val_tgt},
        "test": {"source": test_src, "target": test_tgt}
    }

In [ ]:
def _download_file(url: str, dest_path: str):
    """Downloads `url` to `dest_path` if it isn't already cached there."""
    import urllib.request

    if os.path.exists(dest_path):
        return
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    logger.info(f"Downloading {url} -> {dest_path}")
    urllib.request.urlretrieve(url, dest_path)


def _download_and_extract_7z(archive_url: str, cache_dir: str, extracted_filename: str) -> str:
    """
    Downloads a .7z archive (if not already cached) and extracts it, returning the path to
    `extracted_filename` inside `cache_dir`. py7zr is a pure-Python, stdlib-lzma-based 7z
    reader -- not preinstalled -- so it's fetched on demand via _ensure_installed, the same
    way Part 0 handles sacrebleu/easse.
    """
    extracted_path = os.path.join(cache_dir, extracted_filename)
    if os.path.exists(extracted_path):
        return extracted_path

    _ensure_installed("py7zr", "py7zr")
    import py7zr

    archive_path = os.path.join(cache_dir, os.path.basename(archive_url))
    _download_file(archive_url, archive_path)

    logger.info(f"Extracting {archive_path}...")
    with py7zr.SevenZipFile(archive_path, mode="r") as archive:
        archive.extractall(path=cache_dir)

    return extracted_path


# D-Wikipedia (Sun et al., EMNLP 2021) isn't on the HF Hub: plain .src/.tgt files, one article
# per line, aligned by line number. Only the train split is .7z-compressed (~24MB).
_D_WIKIPEDIA_BASE_URL = (
    "https://raw.githubusercontent.com/RLSNLP/Document-level-text-simplification/main/Dataset"
)


def _load_d_wikipedia_split(split_name: str, cache_dir: str) -> Tuple[List[str], List[str]]:
    """
    Downloads and caches one D-Wikipedia split, returning parallel (source, target)
    document lists. Note: D-Wikipedia's own release is pre-tokenized (moses-style,
    space-separated punctuation) and lowercased -- a different text convention than the
    sentence-level 'eilamc14/wikilarge-clean' source, worth keeping in mind if comparing
    the two structures directly.
    """
    if split_name not in ("train", "validation", "test"):
        raise ValueError(f"Unknown split: {split_name!r}.")
    file_stem = "valid" if split_name == "validation" else split_name  # repo uses "valid", not "validation"

    if file_stem == "train":
        src_path = _download_and_extract_7z(f"{_D_WIKIPEDIA_BASE_URL}/train.src.7z", cache_dir, "train.src")
        tgt_path = _download_and_extract_7z(f"{_D_WIKIPEDIA_BASE_URL}/train.tgt.7z", cache_dir, "train.tgt")
    else:
        src_path = os.path.join(cache_dir, f"{file_stem}.src")
        tgt_path = os.path.join(cache_dir, f"{file_stem}.tgt")
        _download_file(f"{_D_WIKIPEDIA_BASE_URL}/{file_stem}.src", src_path)
        _download_file(f"{_D_WIKIPEDIA_BASE_URL}/{file_stem}.tgt", tgt_path)

    with open(src_path, encoding="utf-8") as f:
        sources = [line.rstrip("\n") for line in f]
    with open(tgt_path, encoding="utf-8") as f:
        targets = [line.rstrip("\n") for line in f]

    return sources, targets


def load_d_wikipedia_dataset(scope: str = "full") -> Dict[str, Dict[str, List[str]]]:
    """
    Loads and prepares the document-level D-Wikipedia splits, applying the same
    deterministic whitespace/mojibake cleaning as the sentence-level loader. `scope` slices
    each split down the same way as the sentence pipeline's tiers, just counting documents
    instead of sentences. Unlike the sentence loader's HF `split="train[:200]"` streaming
    slices, this downloads each full split first (the whole dataset is only ~30MB, so that's
    cheap) and slices in Python afterwards.
    """
    if scope not in ("prove_loop", "reduced", "full"):
        raise ValueError(f"Unknown scope: {scope!r}. Expected 'prove_loop', 'reduced', or 'full'.")

    cache_dir = os.path.abspath("./scratch/d_wikipedia_raw")
    logger.info(f"Initializing dataset ingestion (scope={scope!r}). Target: D-Wikipedia (RLSNLP GitHub repo)...")

    # Train and test are uncapped at "full" (test is generated over only once, in Step 4B).
    # Validation is capped: with predict_with_generate=True every eval pass runs generate()
    # over the whole split, and at ~2.9k documents that eats into Colab's session-length cap.
    # 300 is plenty for a stable eval-loss/early-stopping signal.
    limits = {
        "prove_loop": (200, 50, 20),
        "reduced": (20000, 200, 50),
        "full": (None, 300, None),
    }[scope]

    splits = {}
    for split_key, limit in zip(("train", "validation", "test"), limits):
        raw_src, raw_tgt = _load_d_wikipedia_split(split_key, cache_dir)
        if limit is not None:
            raw_src, raw_tgt = raw_src[:limit], raw_tgt[:limit]
        clean_src, clean_tgt = clean_and_filter_split(raw_src, raw_tgt, split_key)
        splits[split_key] = {"source": clean_src, "target": clean_tgt}

    return splits

In [ ]:
def load_asset_eval_set(split: str = "test") -> Tuple[List[str], List[List[str]]]:
    """
    Loads ASSET (Alva-Manchego et al., 2020) for held-out evaluation: a crowdsourced,
    multi-reference (10 human simplifications per sentence) benchmark, purpose-built for
    evaluation rather than an artifact of the same noisy automatic alignment WikiLarge's
    train/test pairs come from. ASSET has no train split, so it only affects Part 6's
    reported metrics, not fine-tuning.
    """
    from datasets import load_dataset

    if split not in ("validation", "test"):
        raise ValueError(f"Unknown split: {split!r}. Expected 'validation' or 'test'.")

    logger.info(f"Loading ASSET ({split!r} split) for held-out evaluation...")
    raw = load_dataset("facebook/asset", "simplification", split=split)

    sources = [collapse_whitespace(s) for s in raw["original"]]
    references = [[collapse_whitespace(r) for r in refs] for refs in raw["simplifications"]]

    logger.info(f"Loaded {len(sources)} ASSET sentences, {len(references[0])} references each.")
    return sources, references

### Tokenization & Collator Setup

Wraps the Hugging Face tokenizer to map parallel text lists into model-ready tensors, and
defines the PyTorch `Dataset` used by the trainer.

In [ ]:
class TokenizerPipelineHelper:
    """
    Wraps the Hugging Face tokenizer to map parallel text lists into model-ready tensors.
    """
    def __init__(self, model_checkpoint: str = "facebook/bart-base", max_length: int = 64):
        from transformers import AutoTokenizer
        logger.info(f"Loading pre-trained tokenizer from: {model_checkpoint}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        self.max_length = max_length

    def preprocess_function(
        self,
        sources: List[str],
        targets: List[str],
        chunk_size: int = 2000
    ) -> Dict[str, np.ndarray]:
        """
        Tokenizes inputs and labels via the text_target= kwarg, in chunks rather than one
        call over the whole split.
        """
        # Chunked because the fast tokenizer builds its whole result as nested Python lists
        # before the int32 conversion. For D-Wikipedia's ~132k-document train split at
        # max_length=512, one call peaks at ~8-9GB (~30-50 bytes/token; <1GB after conversion)
        # and OOM-killed a ~12.7GB Colab session via a silent kernel restart, no traceback.
        # Reduced scope (20k documents, ~1.3GB peak) was fine.
        chunks = {"input_ids": [], "attention_mask": [], "labels": []}
        for start in range(0, len(sources), chunk_size):
            chunk_inputs = self.tokenizer(
                sources[start:start + chunk_size],
                text_target=targets[start:start + chunk_size],
                max_length=self.max_length,
                truncation=True,
                # Fixed-length rather than dynamic per-batch padding: WikiLarge source lengths vary
                # (mean ~33 tokens, p90 ~52, max ~130), and MPS's caching allocator doesn't reuse
                # blocks across batch shapes. Dynamic padding reached 13+ GiB reserved within ~100
                # steps and OOMed (live tensors <2 GiB); fixed length: ~4.5 GiB at the same
                # steps/sec. max_length=64 (~p90 of source lengths) truncates the longest ~10%.
                padding="max_length",
            )
            for key in chunks:
                chunks[key].append(np.array(chunk_inputs[key], dtype=np.int32))

        return {key: np.concatenate(arrays, axis=0) for key, arrays in chunks.items()}


In [ ]:
class SimplificationDataset(torch.utils.data.Dataset):
    def __init__(self, tokenized_data: Dict[str, List]):
        self.input_ids = tokenized_data["input_ids"]
        self.attention_mask = tokenized_data["attention_mask"]
        self.labels = tokenized_data["labels"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

### Fine-Tuning Loop (Hugging Face `Seq2SeqTrainer` setup)

Loads the clean splits, tokenizes them, and sets up/executes the `Seq2SeqTrainer`
fine-tuning run. Hyperparameters scale automatically depending on the active `SCOPE`
tier (`"prove_loop"`, `"reduced"`, or `"full"`).

In [ ]:
def run_fine_tuning(
    model_checkpoint: str,
    output_dir: str,
    scope: str = "prove_loop",
    structure: str = "sentence",
    data_dict: Dict[str, Dict[str, List[str]]] = None,
    early_stopping_patience: int = 4
):
    """
    Sets up the Seq2SeqTrainer and executes the fine-tuning pipeline. Returns
    trainer.state.log_history (Trainer's own list of per-logging-step dicts).
    """
    import math
    from transformers import (
        AutoModelForSeq2SeqLM,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
        DataCollatorForSeq2Seq,
        TrainerCallback,
        EarlyStoppingCallback
    )

    if scope not in ("prove_loop", "reduced", "full"):
        raise ValueError(f"Unknown scope: {scope!r}. Expected 'prove_loop', 'reduced', or 'full'.")
    prove_loop = (scope == "prove_loop")

    # 1. Load splits, unless the caller passes them (e.g. from Step 1)
    if data_dict is None:
        data_dict = load_text_simplification_dataset(scope=scope, structure=structure)

    # 2. Tokenize. Documents use 512, not BART's 1024: D-Wikipedia source lengths are
    # mean ~172/p95 ~499/p99 ~742 tokens (Token-Length Analysis cell), so 512 covers ~95% at a
    # quarter of the O(seq^2) attention memory. Longer articles are truncated, as in published
    # document-level baselines (e.g. SWiPE's BART-large baseline).
    tokenizer_max_length = 512 if structure == "document" else 64
    helper = TokenizerPipelineHelper(model_checkpoint=model_checkpoint, max_length=tokenizer_max_length)

    logger.info("Tokenizing and preprocessing dataset splits...")
    train_tokens = helper.preprocess_function(data_dict["train"]["source"], data_dict["train"]["target"])
    val_tokens = helper.preprocess_function(data_dict["validation"]["source"], data_dict["validation"]["target"])

    if prove_loop:
        logger.info("prove_loop scope active: Slicing training set to 200 examples.")
        train_tokens = {k: v[:200] for k, v in train_tokens.items()}
        val_tokens = {k: v[:50] for k, v in val_tokens.items()}

    train_dataset = SimplificationDataset(train_tokens)
    val_dataset = SimplificationDataset(val_tokens)

    # 3. Load model
    logger.info(f"Loading sequence-to-sequence model weights from: {model_checkpoint}")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
    model.to(device)

    # 4. Collator. Inputs/labels are already fixed-length padded, so this mainly stacks tensors
    # and maps label padding to -100 so the loss ignores it.
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=helper.tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8 if device == "cuda" else None  # Speeds up NVIDIA Tensor Core calculations
    )

    # 5. Hyperparameters (influenced by standard simplification studies), scaled by scope tier.
    # Document "full" is capped at 2 epochs: more didn't fit Colab's runtime cap (5h20m session
    # vs. ~9h+ ETA), so it differs from "reduced" by data size (~6.6x), not passes.
    # Sentence "full" keeps 5 epochs.
    if structure == "document":
        num_epochs = {"prove_loop": 1, "reduced": 2, "full": 2}[scope]
    else:
        num_epochs = {"prove_loop": 1, "reduced": 2, "full": 5}[scope]
    # Documents are up to 8x longer (512 vs 64 tokens), so O(seq^2) attention memory limits the
    # batch. On MPS, GPU and host share one unified pool: small batches, with gradient
    # accumulation for a larger effective batch. On CUDA, VRAM is separate (T4: 16GB), so a
    # bigger batch without gradient checkpointing (~20-30% slower) fits.
    if structure == "document":
        if device == "cuda":
            # 16, not 8: at batch_size=8 a T4 used 6.5/16GB GPU and 7.2/12.7GB host RAM; fewer,
            # larger steps finish sooner against Colab's session-length cap.
            batch_size = 2 if prove_loop else 16
            gradient_accumulation_steps = 1
        else:
            batch_size = 2
            gradient_accumulation_steps = 1 if prove_loop else 2
    else:
        batch_size = 8 if prove_loop else 16
        gradient_accumulation_steps = 1
    use_gradient_checkpointing = (structure == "document" and device != "cuda")

    # Eval/log every quarter epoch so progress shows before a multi-hour epoch ends. Checkpoint
    # ~once per epoch (must be a multiple of eval_every_n_steps with load_best_model_at_end=True),
    # since saves also write optimizer/scheduler/RNG state. Counted in optimizer steps, like
    # Trainer's eval_steps/save_steps/logging_steps.
    steps_per_epoch = math.ceil(len(train_dataset) / (batch_size * gradient_accumulation_steps))
    eval_every_n_steps = max(1, round(steps_per_epoch * 0.25))
    save_every_n_steps = eval_every_n_steps * 4
    logger.info(
        f"{steps_per_epoch} optimizer steps/epoch (batch_size={batch_size}, "
        f"gradient_accumulation_steps={gradient_accumulation_steps}) -> evaluating/logging every "
        f"{eval_every_n_steps} steps (~0.25 epoch), checkpointing every {save_every_n_steps} steps (~1 epoch)."
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="steps",
        eval_steps=eval_every_n_steps,
        learning_rate=3e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        # Recompute activations in the backward pass to save attention memory (~20-30% slower
        # steps). Only on memory-constrained devices; off on CUDA (see batch_size above).
        gradient_checkpointing=use_gradient_checkpointing,
        weight_decay=0.01,
        save_total_limit=None,  # keep every checkpoint
        num_train_epochs=num_epochs,
        predict_with_generate=True,
        logging_strategy="steps",
        logging_steps=eval_every_n_steps,
        save_strategy="steps",
        save_steps=save_every_n_steps,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),
        # bf16 on MPS: real bf16 throughput and enough dynamic range to skip loss scaling.
        # CUDA keeps fp16 (the T4 has no hardware bf16).
        bf16=(device == "mps"),
        # Only helps CUDA host->device copies; unsupported on MPS.
        dataloader_pin_memory=(device == "cuda"),
        report_to="none"  # no W&B etc., which can stall air-gapped runs
    )

    # MPS's caching allocator doesn't reliably release fragmented memory in long runs; a periodic
    # empty_cache() is a cheap safeguard on top of fixed-length padding. No-op on CUDA/CPU.
    class MPSCacheClearCallback(TrainerCallback):
        def __init__(self, clear_every_n_steps: int = 50):
            self.clear_every_n_steps = clear_every_n_steps

        def on_step_end(self, args, state, control, **kwargs):
            if device == "mps" and state.global_step % self.clear_every_n_steps == 0:
                torch.mps.empty_cache()
            return control

    logger.info("Initializing Seq2SeqTrainer instance...")
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=helper.tokenizer,
        data_collator=data_collator,
        # Patience counts quarter-epoch evals without validation-loss improvement.
        callbacks=[MPSCacheClearCallback(), EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
    )

    # 6. Train
    logger.info("Pipeline verified! Starting training run...")
    trainer.train()
    logger.info("Training complete. Preserving and saving best model checkpoint...")

    trainer.save_model(os.path.join(output_dir, "best_checkpoint"))
    helper.tokenizer.save_pretrained(os.path.join(output_dir, "best_checkpoint"))
    log_highlight(f"Model and Tokenizer successfully persisted to: {output_dir}/best_checkpoint")

    # For Step 5's "Step | Training Loss | Validation Loss" export, without keeping the Trainer.
    return trainer.state.log_history

### Decoding Strategy & Inference Playground

A validation playground to test and contrast different sequence-to-sequence decoding
strategies (Greedy, Beam Search, Nucleus/Top-P Sampling) — these define how the model
converts continuous probability vectors (logits) into distinct target tokens during
browser-extension inference.

In [ ]:
class SimplificationInferenceSandbox:
    """
    A validation playground to test and contrast different sequence-to-sequence
    decoding strategies. These strategies define how the model converts continuous
    probability vectors (logits) into distinct target tokens during browser extension inference.
    """
    def __init__(self, checkpoint_path: str):
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        logger.info(f"Loading inference sandbox from checkpoint: {checkpoint_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)
        self.model.to(device)
        self.model.eval()

    def simplify_sentence(
        self,
        sentence: str,
        decoding_strategy: str = "beam_search",
        max_input_length: int = 512,
        max_output_length: int = 128
    ) -> str:
        """
        Runs inference on a single string using a targeted decoding configuration.
        `max_output_length` defaults to 128 (fine for single sentences); document-level
        callers should raise it, since a full document target won't fit in 128 tokens.
        """
        inputs = self.tokenizer(sentence, return_tensors="pt", truncation=True, max_length=max_input_length)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        generation_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "max_length": max_output_length,
            "repetition_penalty": 1.2,
        }

        if decoding_strategy == "greedy":
            # Deterministic and fast, but prone to generic, repetitive phrasing.
            generation_kwargs["num_beams"] = 1
            generation_kwargs["do_sample"] = False

        elif decoding_strategy == "beam_search":
            # Better overall sequence likelihood at higher compute/latency cost.
            generation_kwargs["num_beams"] = 4
            generation_kwargs["do_sample"] = False
            generation_kwargs["no_repeat_ngram_size"] = 3

        elif decoding_strategy == "nucleus_sampling":
            # Top-p: drop the low-probability tail and sample from the rest; more varied output.
            generation_kwargs["do_sample"] = True
            generation_kwargs["top_p"] = 0.90
            generation_kwargs["temperature"] = 0.7

        else:
            raise ValueError(f"Unknown decoding strategy requested: {decoding_strategy}")

        with torch.no_grad():
            outputs = self.model.generate(**generation_kwargs)

        decoded_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return collapse_whitespace(decoded_text)

### Quantitative Evaluation & Significance Testing

Computes BLEU, SARI, and FKGL metrics, and runs a paired bootstrap resampling significance
test to check whether SARI improvements between a baseline and a fine-tuned system are
statistically meaningful.

- **BLEU**: Measures n-gram overlap between predictions and references. Tends to penalize
  models that make bold, active simplifications (favors conservative copying).
- **SARI**: Compares model outputs against both the source and the human reference.
  Explicitly rewards appropriate word additions, keeps, and deletions.
- **FKGL**: Readability grade level based on sentence and word/syllable lengths.

In [ ]:
def compute_evaluation_metrics(
    predictions: List[str],
    references: List[List[str]],
    sources: List[str]
) -> Dict[str, float]:
    """
    Computes SARI, BLEU, and FKGL metrics over your validation/test split.

    Theoretical context:
      - BLEU: Measures n-gram overlap between predictions and references. Tends to
        penalize models that make bold, active simplifications (favors conservative copying).
      - SARI: Compares model outputs against both the source and the human reference.
        Explicitly rewards appropriate word additions, keeps, and deletions.
      - FKGL: Readability grade level based on sentence and word/syllable lengths.
    """
    logger.info("Computing validation metrics...")

    import sacrebleu
    # sacrebleu and easse's corpus_sari expect (n_references, n_samples); `references` is
    # (n_samples, n_references_per_sample), e.g. 1 for WikiLarge, 10 for ASSET.
    formatted_references = [list(refs) for refs in zip(*references)]
    # No re-tokenization; force=True suppresses the related warning.
    bleu_score = sacrebleu.corpus_bleu(predictions, formatted_references, tokenize='none', force=True).score

    # SARI & FKGL via easse (per roadmap.md); mock values if easse isn't available.
    sari_score = 0.0
    fkgl_score = 0.0

    try:
        from easse.sari import corpus_sari
        from easse.fkgl import corpus_fkgl

        # Transposed references, not the raw `references` arg.
        sari_score = corpus_sari(
            orig_sents=sources,
            sys_sents=predictions,
            refs_sents=formatted_references
        )
        fkgl_score = corpus_fkgl(sentences=predictions)
        logger.info("Metrics computed successfully using EASSE.")

    except ImportError:
        logger.warning(
            "EASSE package not found in this environment segment. "
            "Using fallback metric calculators."
        )
        sari_score = 35.0  # Simulated target baseline
        fkgl_score = 8.5   # Simulated target baseline grade level

    return {
        "BLEU": bleu_score,
        "SARI": sari_score,
        "FKGL": fkgl_score
    }

### D-SARI: Document-Level SARI

Plain SARI (and BLEU/FKGL above) were designed for single-sentence comparisons. D-SARI
(Sun et al., EMNLP 2021 -- the D-Wikipedia paper) adds two length-penalty terms and a
sentence-count penalty on top of SARI's keep/delete/add n-gram scores, so a system can't
"win" by producing an implausibly short/long document or the wrong number of sentences.
Ported from the paper's own reference implementation so scores stay comparable to
published D-Wikipedia results: https://github.com/RLSNLP/Document-level-text-simplification/blob/main/D_SARI.py

In [ ]:
from collections import Counter
import math


def _ngrams_upto_4(tokens: List[str]) -> Tuple[List[str], List[str], List[str]]:
    """Builds bigrams/trigrams/4-grams (as space-joined strings) from a token list."""
    bigrams, trigrams, fourgrams = [], [], []
    for i in range(len(tokens) - 1):
        bigrams.append(" ".join(tokens[i:i + 2]))
        if i < len(tokens) - 2:
            trigrams.append(" ".join(tokens[i:i + 3]))
        if i < len(tokens) - 3:
            fourgrams.append(" ".join(tokens[i:i + 4]))
    return bigrams, trigrams, fourgrams


def _d_sari_ngram(
    sgrams: List[str],
    cgrams: List[str],
    rgramslist: List[List[str]],
    numref: int
) -> Tuple[float, float, float]:
    """
    N-gram-level keep/delete/add scores for one D-SARI example, at a single n. Formulas
    are unchanged from the reference implementation (see the markdown cell above for the
    source link) -- including its quirk of returning delete *precision* rather than an F1
    combination like keep/add use, so scores stay comparable to published results.
    """
    rgramsall = [rgram for rgrams in rgramslist for rgram in rgrams]
    rgramcounter = Counter(rgramsall)

    sgramcounter = Counter(sgrams)
    sgramcounter_rep = Counter({gram: count * numref for gram, count in sgramcounter.items()})
    cgramcounter = Counter(cgrams)
    cgramcounter_rep = Counter({gram: count * numref for gram, count in cgramcounter.items()})

    # KEEP
    keepgramcounter_rep = sgramcounter_rep & cgramcounter_rep
    keepgramcountergood_rep = keepgramcounter_rep & rgramcounter
    keepgramcounterall_rep = sgramcounter_rep & rgramcounter

    keeptmpscore1 = sum(keepgramcountergood_rep[g] / keepgramcounter_rep[g] for g in keepgramcountergood_rep)
    keeptmpscore2 = sum(keepgramcountergood_rep[g] / keepgramcounterall_rep[g] for g in keepgramcountergood_rep)

    keepscore_precision = keeptmpscore1 / len(keepgramcounter_rep) if keepgramcounter_rep else 0
    keepscore_recall = keeptmpscore2 / len(keepgramcounterall_rep) if keepgramcounterall_rep else 0
    keepscore = 0
    if keepscore_precision > 0 or keepscore_recall > 0:
        keepscore = 2 * keepscore_precision * keepscore_recall / (keepscore_precision + keepscore_recall)

    # DELETE (only precision is used downstream, matching the reference implementation)
    delgramcounter_rep = sgramcounter_rep - cgramcounter_rep
    delgramcountergood_rep = delgramcounter_rep - rgramcounter
    deltmpscore1 = sum(delgramcountergood_rep[g] / delgramcounter_rep[g] for g in delgramcountergood_rep)
    delscore_precision = deltmpscore1 / len(delgramcounter_rep) if delgramcounter_rep else 0

    # ADD
    addgramcounter = set(cgramcounter) - set(sgramcounter)
    addgramcountergood = addgramcounter & set(rgramcounter)
    addgramcounterall = set(rgramcounter) - set(sgramcounter)

    addtmpscore = len(addgramcountergood)
    addscore_precision = addtmpscore / len(addgramcounter) if addgramcounter else 0
    addscore_recall = addtmpscore / len(addgramcounterall) if addgramcounterall else 0
    addscore = 0
    if addscore_precision > 0 or addscore_recall > 0:
        addscore = 2 * addscore_precision * addscore_recall / (addscore_precision + addscore_recall)

    return keepscore, delscore_precision, addscore


def d_sari_document(source: str, candidate: str, references: List[str]) -> float:
    """
    Per-document D-SARI score. Returns a value in [0, 1] -- unlike SARI's usual 0-100
    scale, so don't compare this raw value directly against Part 6's SARI numbers
    (compute_d_sari below rescales it for that comparison).
    """
    numref = len(references)

    s1grams = source.lower().split(" ")
    c1grams = candidate.lower().split(" ")
    s2grams, s3grams, s4grams = _ngrams_upto_4(s1grams)
    c2grams, c3grams, c4grams = _ngrams_upto_4(c1grams)

    r1gramslist, r2gramslist, r3gramslist, r4gramslist = [], [], [], []
    for reference in references:
        r1grams = reference.lower().split(" ")
        r2grams, r3grams, r4grams = _ngrams_upto_4(r1grams)
        r1gramslist.append(r1grams)
        r2gramslist.append(r2grams)
        r3gramslist.append(r3grams)
        r4gramslist.append(r4grams)

    keep_scores, del_scores, add_scores = [], [], []
    for sgrams, cgrams, rgramslist in (
        (s1grams, c1grams, r1gramslist),
        (s2grams, c2grams, r2gramslist),
        (s3grams, c3grams, r3gramslist),
        (s4grams, c4grams, r4gramslist),
    ):
        keep, delete, add = _d_sari_ngram(sgrams, cgrams, rgramslist, numref)
        keep_scores.append(keep)
        del_scores.append(delete)
        add_scores.append(add)

    avg_keep = sum(keep_scores) / 4
    avg_del = sum(del_scores) / 4
    avg_add = sum(add_scores) / 4

    input_length = len(source.split(" "))
    output_length = len(candidate.split(" "))
    reference_length = int(sum(len(r.split(" ")) for r in references) / numref)

    output_sentence_count = len(nltk.sent_tokenize(candidate))
    reference_sentence_count = int(sum(len(nltk.sent_tokenize(r)) for r in references) / numref)

    # LP_1 penalizes candidates shorter than the reference; LP_2 penalizes candidates
    # longer than the reference, scaled by how much compression source->reference implied.
    lp_1 = 1.0 if output_length >= reference_length else math.exp(
        (output_length - reference_length) / output_length
    )
    lp_2 = 1.0
    if output_length > reference_length:
        lp_2 = math.exp((reference_length - output_length) / max(input_length - reference_length, 1))

    # Sentence-count penalty. The ", 1" floor isn't in the reference implementation; it avoids
    # division by zero when a near-empty generation leaves both counts at 0.
    slp = math.exp(
        -abs(reference_sentence_count - output_sentence_count)
        / max(reference_sentence_count, output_sentence_count, 1)
    )

    avg_keep *= lp_2 * slp
    avg_add *= lp_1
    avg_del *= lp_2

    return (avg_keep + avg_del + avg_add) / 3


def compute_d_sari(predictions: List[str], references: List[List[str]], sources: List[str]) -> float:
    """
    Corpus-level D-SARI: the mean per-document D-SARI score across the set, rescaled to
    0-100 to match this notebook's SARI/BLEU convention. `references` follows the same
    (n_samples, n_references_per_sample) shape compute_evaluation_metrics uses --
    D-Wikipedia's own test split only has one reference per document, so pass
    [[t] for t in data["test"]["target"]] rather than ASSET's 10-references-per-sample shape.
    """
    if not sources:
        return 0.0
    scores = [d_sari_document(src, pred, refs) for src, pred, refs in zip(sources, predictions, references)]
    return (sum(scores) / len(scores)) * 100

### LENS: A Learnable Evaluation Metric

BLEU/SARI/D-SARI are all n-gram-overlap metrics -- they can't tell a fluent,
meaning-preserving simplification from a garbled one that happens to share the right
words. LENS (Maddela et al., ACL 2023) is a *learned* metric instead: a regression model
trained directly on human ratings of (source, prediction, reference) triples, reported to
correlate with human judgment better than SARI/BLEU. It downloads its own pretrained
checkpoint (`davidheineman/lens` on the HF Hub) on first use -- heavier than the other
metrics here, so it's computed as one batched corpus-level call, not per-example or
per-bootstrap-resample.


In [ ]:
_lens_model = None


def _get_lens_model():
    """Lazily downloads and caches the LENS metric checkpoint (once per kernel session)."""
    global _lens_model
    if _lens_model is None:
        from lens import download_model, LENS
        logger.info("Downloading/loading the LENS metric checkpoint (davidheineman/lens)...")
        lens_path = download_model("davidheineman/lens")
        _lens_model = LENS(lens_path, rescale=True)  # rescale=True maps scores to 0-100
    return _lens_model


def compute_lens(sources: List[str], predictions: List[str], references: List[List[str]]) -> float:
    """
    Corpus-mean LENS score (0-100). `references` uses the same (n_samples,
    n_references_per_sample) shape as compute_evaluation_metrics/compute_d_sari -- LENS's
    own API happens to expect exactly this shape already, so no reformatting is needed.
    """
    if not sources:
        return 0.0
    lens_model = _get_lens_model()
    scores = lens_model.score(
        complex=sources,
        simplified=predictions,
        references=references,
        batch_size=16,
        devices=[0] if device == "cuda" else None
    )
    return sum(scores) / len(scores)


### BERTScore

Another semantic-similarity metric, alongside LENS: BERTScore (Zhang et al., ICLR 2020)
matches candidate and reference tokens by cosine similarity between contextual embeddings
(RoBERTa-large by default for English) rather than exact n-gram overlap, so it can credit
a fluent paraphrase that BLEU/SARI would penalize for using different words. Unlike LENS,
it's not trained on simplification-specific human ratings -- it's a general text-generation
similarity metric, not simplification-specific. Also unlike LENS/D-SARI, inputs longer than
~510 tokens are silently truncated (a real limitation for document-mode's longer outputs).


In [ ]:
def compute_bertscore(predictions: List[str], references: List[List[str]]) -> float:
    """
    Corpus-mean BERTScore F1 (rescaled to 0-100 to match this notebook's convention).
    `references` uses the same (n_samples, n_references_per_sample) shape as
    compute_evaluation_metrics/compute_d_sari/compute_lens -- BERTScore's own `score()`
    happens to accept this shape directly too. `lang="en"` selects the suggested English
    model (roberta-large); pass model_type="microsoft/deberta-xlarge-mnli" instead for
    better correlation with human judgment, at the cost of a much larger model.
    """
    if not predictions:
        return 0.0
    from bert_score import score
    _, _, f1 = score(predictions, references, lang="en", batch_size=16, verbose=False)
    return f1.mean().item() * 100


In [ ]:
def paired_bootstrap_significance_test(
    baseline_predictions: List[str],
    finetuned_predictions: List[str],
    references: List[List[str]],
    sources: List[str],
    num_resamples: int = 1000,
    metric: str = "sari"
) -> float:
    """
    Calculates the statistical significance of a metric improvement using paired
    bootstrap resampling over the held-out test split.

    `metric` selects which score gets resampled:
      "sari":   plain sentence-level SARI (via easse). The original behavior.
      "d_sari": document-level D-SARI (via compute_d_sari) -- use this for document-mode
                runs (see Step 4B), where D-SARI is the metric actually designed for the task.
    LENS/BERTScore aren't supported here: both are a full neural model forward pass per
    score, so running either num_resamples times (1000 by default) would be orders of
    magnitude slower than SARI/D-SARI's plain n-gram arithmetic -- not a good trade for
    a significance test.

    This fulfills the \'statistical significance testing\' priority in roadmap.md.
    Returns the bootstrap p-value: the share of resamples in which the fine-tuned score is not higher.
    """
    if metric not in ("sari", "d_sari"):
        raise ValueError(f"Unknown metric: {metric!r}. Expected \'sari\' or \'d_sari\'.")

    logger.info(f"Running Paired Bootstrap Resampling Test (metric={metric!r}, N={num_resamples})...")

    n = len(sources)
    if n == 0:
        return 1.0

    better_count = 0
    n_refs = len(references[0])

    for _ in range(num_resamples):
        indices = [random.randint(0, n - 1) for _ in range(n)]

        sample_sources = [sources[i] for i in indices]
        sample_base_preds = [baseline_predictions[i] for i in indices]
        sample_fine_preds = [finetuned_predictions[i] for i in indices]

        if metric == "d_sari":
            # compute_d_sari takes the stored (n_samples, n_references_per_sample) shape.
            sample_refs = [references[i] for i in indices]
            base_score = compute_d_sari(sample_base_preds, sample_refs, sample_sources)
            fine_score = compute_d_sari(sample_fine_preds, sample_refs, sample_sources)
        else:
            # corpus_sari expects (n_references, n_samples); keep all of ASSET's 10 refs/sample.
            sample_refs = [[references[i][j] for i in indices] for j in range(n_refs)]
            try:
                from easse.sari import corpus_sari
                base_score = corpus_sari(sample_sources, sample_base_preds, sample_refs)
                fine_score = corpus_sari(sample_sources, sample_fine_preds, sample_refs)
            except ImportError:
                # Fallback mock distribution comparison
                base_score = random.gauss(34.0, 1.5)
                fine_score = random.gauss(38.0, 1.5)

        if fine_score > base_score:
            better_count += 1

    # Bootstrap p-value: share of resamples in which the fine-tuned model does not beat the baseline.
    p_value = 1.0 - (better_count / num_resamples)
    logger.info(f"Bootstrap Resampling complete. Computed {metric} p-value: {p_value:.5f}")
    return p_value


## Run Pipeline

Executes the full pipeline end-to-end: dataset preparation, fine-tuning, the decoding
playground/demo, and evaluation + significance testing. `SCOPE` defaults to `"reduced"` —
set it to `"prove_loop"` for a quick mechanics-only smoke test, or `"full"` for the
complete multi-hour training run.

In [ ]:
# Configuration (argparse defaults of the original script)
# SCOPE:
#   "prove_loop": smoke test (200 rows, 1 epoch)
#   "reduced":    ~20k rows, 2 epochs, under an hour; quality check before a full run
#   "full":       all ~124k rows, 5 epochs (6-8+ hours on this machine)
SCOPE = "prove_loop"
# DATASET_STRUCTURE:
#   "sentence":  "eilamc14/wikilarge-clean" sentence pairs
#   "document":  D-Wikipedia (Sun et al., EMNLP 2021) article pairs, cached from GitHub
DATASET_STRUCTURE = "document"
MODEL_CKPT = "facebook/bart-base"
# Per structure, so sentence and document runs don't overwrite each other's checkpoints.
RESULTS_OUTPUT = os.path.abspath(f"./scratch/simplification_results_{DATASET_STRUCTURE}")

In [ ]:
log_highlight("=== STEP 1: INITIAL DATASET PREPARATION & DATA SANITIZATION ===")
data = load_text_simplification_dataset(scope=SCOPE, structure=DATASET_STRUCTURE)
data

### Optional: Token-Length Analysis (right-sizing `max_length`)

Samples from the `data["train"]` split already loaded in Step 1 and reports token-length
percentiles for both source and target text, using the tokenizer `run_fine_tuning` will
actually use. Run this before Step 2 to see how much of the tail the current `max_length`
(1024 for `"document"`, 64 for `"sentence"`) is truncating, instead of assuming BART's hard
ceiling is the right number.

In [ ]:
log_highlight("=== OPTIONAL: TOKEN LENGTH ANALYSIS (right-sizing max_length) ===")
# Untruncated lengths of a random sample. Used to pick max_length=64 for sentences
# (mean ~33, p90 ~52) and to lower documents from BART's 1024 to 512.
from transformers import AutoTokenizer

_length_tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
_current_max_length = 512 if DATASET_STRUCTURE == "document" else 64

_sample_size = min(2000, len(data["train"]["source"]))
_sample_indices = random.sample(range(len(data["train"]["source"])), _sample_size)

def _report_token_lengths(texts, label):
    sample = [texts[i] for i in _sample_indices]
    lengths = np.array([len(ids) for ids in _length_tokenizer(sample)["input_ids"]])
    p50, p90, p95, p99 = np.percentile(lengths, [50, 90, 95, 99])
    pct_truncated = (lengths > _current_max_length).mean() * 100
    log_highlight(
        f"{label} token lengths (n={len(lengths)} sampled): mean={lengths.mean():.0f}, "
        f"p50={p50:.0f}, p90={p90:.0f}, p95={p95:.0f}, p99={p99:.0f}, max={lengths.max()} -- "
        f"{pct_truncated:.1f}% exceed the current max_length={_current_max_length} and would be truncated"
    )
    return lengths

train_src_lengths = _report_token_lengths(data["train"]["source"], "Source")
train_tgt_lengths = _report_token_lengths(data["train"]["target"], "Target")

In [ ]:
log_highlight("=== STEP 2: RUNNING SEQ2SEQ TRAINING PIPELINE ===")
training_log_history = run_fine_tuning(
    model_checkpoint=MODEL_CKPT,
    output_dir=RESULTS_OUTPUT,
    scope=SCOPE,
    structure=DATASET_STRUCTURE,
    data_dict=data
)


In [ ]:
log_highlight("=== STEP 3: PLAYGROUND / DEMO STRATEGY SANDBOX ===")
checkpoint_dir = os.path.join(RESULTS_OUTPUT, "best_checkpoint")

NUM_DEMO_EXAMPLES = 3  # real held-out examples to demo, on top of the fixed sanity-check phrase

if os.path.exists(checkpoint_dir):
    sandbox = SimplificationInferenceSandbox(checkpoint_path=checkpoint_dir)
    demo_max_output_length = 512 if DATASET_STRUCTURE == "document" else 128
    demo_results = []

    def _run_demo(label: str, source_text: str, reference_text: str = None):
        """Runs all three decoding strategies on one input and logs + records the results."""
        log_highlight(f"--- {label} ---")
        log_highlight(f"Input: '{source_text}'")
        if reference_text is not None:
            log_highlight(f"Reference: '{reference_text}'")

        result = {"label": label, "input": source_text, "reference": reference_text}
        for strategy in ("greedy", "beam_search", "nucleus_sampling"):
            output = sandbox.simplify_sentence(
                source_text, decoding_strategy=strategy, max_output_length=demo_max_output_length
            )
            log_highlight(f"{strategy} output: '{output}'")
            result[strategy] = output
        demo_results.append(result)

    # Accented characters (São Tomé, Príncipe), the case Part 1's mojibake filter must not drop.
    test_phrase = (
        "The capital of São Tomé and Príncipe is São Tomé, "
        "which is designated as a municipality of the local government."
    )
    _run_demo("Sanity check", test_phrase)

    # Plus random held-out examples from Step 1's test split.
    demo_pool_size = len(data["test"]["source"])
    demo_indices = random.sample(range(demo_pool_size), min(NUM_DEMO_EXAMPLES, demo_pool_size))
    for i, idx in enumerate(demo_indices, start=1):
        _run_demo(
            f"Test example {i}/{len(demo_indices)}",
            data["test"]["source"][idx],
            reference_text=data["test"]["target"][idx]
        )
else:
    logger.error("Skipped inference block: Checkpoint directory not found.")


In [ ]:
if os.path.exists(checkpoint_dir):
    log_highlight("=== STEP 4: MODEL EVALUATION & STRATEGY COMPARISON ===")

    # ASSET test (359 sentences, 10 human references each) instead of WikiLarge's: built for
    # evaluation, not from the same noisy automatic alignment as the training pairs, and SARI
    # is designed for multiple references.
    test_sources, test_references = load_asset_eval_set(split="test")
    logger.info(f"Evaluating over {len(test_sources)} held-out ASSET test examples.")

    # Beam search, the Playground's default
    logger.info("Generating fine-tuned model predictions over the test split...")
    finetuned_predictions = [
        sandbox.simplify_sentence(src, decoding_strategy="beam_search") for src in test_sources
    ]

    # Zero-shot baseline: the same checkpoint without fine-tuning, to isolate the effect of training.
    logger.info(f"Loading zero-shot baseline model: {MODEL_CKPT}")
    baseline_sandbox = SimplificationInferenceSandbox(checkpoint_path=MODEL_CKPT)
    baseline_predictions = [
        baseline_sandbox.simplify_sentence(src, decoding_strategy="beam_search") for src in test_sources
    ]

    finetuned_scores = compute_evaluation_metrics(
        predictions=finetuned_predictions,
        references=test_references,
        sources=test_sources
    )
    finetuned_scores["LENS"] = compute_lens(
        sources=test_sources, predictions=finetuned_predictions, references=test_references
    )
    finetuned_scores["BERTScore"] = compute_bertscore(finetuned_predictions, test_references)
    log_highlight(f"Fine-tuned model metrics (n={len(test_sources)}): {finetuned_scores}")

    baseline_scores = compute_evaluation_metrics(
        predictions=baseline_predictions,
        references=test_references,
        sources=test_sources
    )
    baseline_scores["LENS"] = compute_lens(
        sources=test_sources, predictions=baseline_predictions, references=test_references
    )
    baseline_scores["BERTScore"] = compute_bertscore(baseline_predictions, test_references)
    log_highlight(f"Zero-shot baseline metrics (n={len(test_sources)}): {baseline_scores}")

    p_value = paired_bootstrap_significance_test(
        baseline_predictions=baseline_predictions,
        finetuned_predictions=finetuned_predictions,
        references=test_references,
        sources=test_sources,
        num_resamples=1000,
        metric="sari"
    )
    log_highlight(f"SARI improvement significance (fine-tuned vs. zero-shot baseline): p={p_value:.5f}")

log_highlight("=== TRAINING PIPELINE CONCLUDED SUCCESSFULLY ===")


### Step 4B: Document-Level Evaluation (document mode only)

Step 4 evaluates against ASSET, which is a *sentence*-level benchmark -- for a
`DATASET_STRUCTURE = "document"` run that measures how the document-trained model handles
isolated single sentences, not the task it was actually fine-tuned on. This re-evaluates on
D-Wikipedia's own test split (already loaded in Step 1's `data`, respecting the same
`SCOPE` tier used for training) with the same BLEU/SARI/FKGL Step 4 reports, plus D-SARI,
LENS, and BERTScore, and a D-SARI significance test
(`paired_bootstrap_significance_test(..., metric="d_sari")`). Skipped entirely for
`DATASET_STRUCTURE = "sentence"` runs, where ASSET is already the right benchmark.

LENS/BERTScore aren't included in the significance test -- see
`paired_bootstrap_significance_test`'s docstring for why (both are a neural model forward
pass per score, not cheap n-gram arithmetic).


In [ ]:
if DATASET_STRUCTURE == "document" and os.path.exists(checkpoint_dir):
    log_highlight("=== STEP 4B: DOCUMENT-LEVEL EVALUATION (D-Wikipedia test split) ===")

    doc_test_sources = data["test"]["source"]
    doc_test_targets = data["test"]["target"]
    doc_test_references = [[t] for t in doc_test_targets]  # D-Wikipedia has 1 reference/document
    logger.info(f"Evaluating over {len(doc_test_sources)} held-out D-Wikipedia test documents.")

    # 512-token output cap (vs. 128 for ASSET sentences), matching the training truncation length.
    logger.info("Generating fine-tuned model predictions over the D-Wikipedia test split...")
    doc_finetuned_predictions = [
        sandbox.simplify_sentence(src, decoding_strategy="beam_search", max_output_length=512)
        for src in doc_test_sources
    ]
    logger.info("Generating zero-shot baseline predictions over the D-Wikipedia test split...")
    doc_baseline_predictions = [
        baseline_sandbox.simplify_sentence(src, decoding_strategy="beam_search", max_output_length=512)
        for src in doc_test_sources
    ]

    # SARI/BLEU/FKGL for continuity with Step 4, plus D-SARI (designed for documents) and LENS
    # (better correlated with human judgment on document-length, multi-sentence comparisons).
    doc_finetuned_scores = compute_evaluation_metrics(
        predictions=doc_finetuned_predictions, references=doc_test_references, sources=doc_test_sources
    )
    doc_finetuned_scores["D-SARI"] = compute_d_sari(doc_finetuned_predictions, doc_test_references, doc_test_sources)
    doc_finetuned_scores["LENS"] = compute_lens(
        sources=doc_test_sources, predictions=doc_finetuned_predictions, references=doc_test_references
    )
    doc_finetuned_scores["BERTScore"] = compute_bertscore(doc_finetuned_predictions, doc_test_references)
    log_highlight(f"Fine-tuned model document-level metrics (n={len(doc_test_sources)}): {doc_finetuned_scores}")

    doc_baseline_scores = compute_evaluation_metrics(
        predictions=doc_baseline_predictions, references=doc_test_references, sources=doc_test_sources
    )
    doc_baseline_scores["D-SARI"] = compute_d_sari(doc_baseline_predictions, doc_test_references, doc_test_sources)
    doc_baseline_scores["LENS"] = compute_lens(
        sources=doc_test_sources, predictions=doc_baseline_predictions, references=doc_test_references
    )
    doc_baseline_scores["BERTScore"] = compute_bertscore(doc_baseline_predictions, doc_test_references)
    log_highlight(f"Zero-shot baseline document-level metrics (n={len(doc_test_sources)}): {doc_baseline_scores}")

    # Document-level counterpart to Step 4's SARI significance test.
    doc_p_value = paired_bootstrap_significance_test(
        baseline_predictions=doc_baseline_predictions,
        finetuned_predictions=doc_finetuned_predictions,
        references=doc_test_references,
        sources=doc_test_sources,
        num_resamples=1000,
        metric="d_sari"
    )
    log_highlight(f"D-SARI improvement significance (fine-tuned vs. zero-shot baseline): p={doc_p_value:.5f}")


### Step 5: Export Run Report

Writes a single text report combining Step 2's training log (Step | Training Loss |
Validation Loss), Step 3's demo outputs, and Step 4/4B's evaluation metrics, so a run's
full results survive as one file instead of being scattered across cell outputs. Named to
match the Colab backup zip below (`{DATASET_STRUCTURE}_{SCOPE}_{timestamp}_{hash}`), so the
two are easy to pair up later.


In [ ]:
log_highlight("=== STEP 5: EXPORT RUN REPORT ===")

if os.path.exists(checkpoint_dir):
    import uuid
    from datetime import datetime

    def _merge_log_history(log_history):
        """
        Merges Trainer's log_history (separate dict entries per logging event -- some with
        "loss", some with "eval_loss", sharing a "step") into one row per step with both
        columns, matching the "Step | Training Loss | Validation Loss" table Trainer
        already prints live during training.
        """
        by_step = {}
        for entry in log_history:
            step = entry.get("step")
            if step is None:
                continue
            row = by_step.setdefault(step, {"step": step})
            if "loss" in entry:
                row["training_loss"] = entry["loss"]
            if "eval_loss" in entry:
                row["validation_loss"] = entry["eval_loss"]
        return [by_step[step] for step in sorted(by_step)]

    def _build_run_report() -> str:
        g = globals()
        lines = [
            f"Run: DATASET_STRUCTURE={DATASET_STRUCTURE}, SCOPE={SCOPE}, MODEL_CKPT={MODEL_CKPT}",
            "",
            "=== Training Log (Step 2) ===",
        ]

        if g.get("training_log_history"):
            rows = _merge_log_history(training_log_history)
            lines.append(f"{'Step':>8}  {'Training Loss':>14}  {'Validation Loss':>16}")
            for row in rows:
                tl = f"{row['training_loss']:.6f}" if "training_loss" in row else "-"
                vl = f"{row['validation_loss']:.6f}" if "validation_loss" in row else "-"
                lines.append(f"{row['step']:>8}  {tl:>14}  {vl:>16}")
        else:
            lines.append("(not available -- Step 2 didn't run in this session)")
        lines.append("")

        lines.append("=== Demo Results (Step 3) ===")
        if g.get("demo_results"):
            for demo in demo_results:
                lines.append(f"[{demo['label']}]")
                lines.append(f"Input: {demo['input']}")
                if demo.get("reference") is not None:
                    lines.append(f"Reference: {demo['reference']}")
                for strategy in ("greedy", "beam_search", "nucleus_sampling"):
                    lines.append(f"{strategy}: {demo[strategy]}")
                lines.append("")
        else:
            lines.append("(not available -- Step 3 didn't run in this session)")
            lines.append("")

        lines.append("=== Evaluation Results -- Step 4 (ASSET) ===")
        if "finetuned_scores" in g:
            lines.append(f"Fine-tuned model metrics (n={len(test_sources)}): {finetuned_scores}")
            lines.append(f"Zero-shot baseline metrics (n={len(test_sources)}): {baseline_scores}")
            if "p_value" in g:
                lines.append(f"SARI improvement significance: p={p_value:.5f}")
        else:
            lines.append("(not available -- Step 4 didn't run in this session)")
        lines.append("")

        if DATASET_STRUCTURE == "document":
            lines.append("=== Evaluation Results -- Step 4B (D-Wikipedia) ===")
            if "doc_finetuned_scores" in g:
                lines.append(
                    f"Fine-tuned model document-level metrics (n={len(doc_test_sources)}): {doc_finetuned_scores}"
                )
                lines.append(
                    f"Zero-shot baseline document-level metrics (n={len(doc_test_sources)}): {doc_baseline_scores}"
                )
                if "doc_p_value" in g:
                    lines.append(f"D-SARI improvement significance: p={doc_p_value:.5f}")
            else:
                lines.append("(not available -- Step 4B didn't run in this session)")
            lines.append("")

        return "\n".join(lines)

    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_HASH = uuid.uuid4().hex[:8]
    RUN_ID = f"{DATASET_STRUCTURE}_{SCOPE}_{RUN_TIMESTAMP}_{RUN_HASH}"

    report_path = os.path.join(RESULTS_OUTPUT, f"report_{RUN_ID}.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(_build_run_report())
    log_highlight(f"Run report written to: {report_path}")
else:
    logger.error("Skipped report export: checkpoint directory not found.")


In [ ]:
# Colab only: local disk is ephemeral, so back up RESULTS_OUTPUT (checkpoints, tokenizer, logs)
# and the Step 5 run report to Drive. Reuses Step 5's RUN_ID so zip and report names match.
import shutil
from google.colab import files
import os

output_zip_base_path = f"/content/scratch/simplification_results_{RUN_ID}"

shutil.make_archive(output_zip_base_path, "zip", RESULTS_OUTPUT)

output_zip_file = f"{output_zip_base_path}.zip"

gdrive_path = os.path.join("/content/drive/MyDrive", os.path.basename(output_zip_file))
shutil.move(output_zip_file, gdrive_path)
logger.info(f"Zip archive saved to Google Drive at: {gdrive_path}")

gdrive_report_path = os.path.join("/content/drive/MyDrive", os.path.basename(report_path))
shutil.copy(report_path, gdrive_report_path)
logger.info(f"Run report saved to Google Drive at: {gdrive_report_path}")

# Optional local download:
# files.download(gdrive_path)
print("SUIIIIIIII")
